In [11]:
import json

In [12]:
def format_calculation_flag(flag):
    """是否涉及运算 → 计算题/非计算题"""
    if "不涉及" in flag:
        return "非计算题"
    else:
        return "计算题"

In [13]:
# 提取选项
def format_options(options_str):
    # 按 A. B. C. D. 切分
    parts = []
    current = ""
    for token in options_str.split(" "):
        if token.startswith(("A.", "B.", "C.", "D.")):
            if current:
                parts.append(current.strip())
            current = token
        else:
            current += " " + token
    if current:
        parts.append(current.strip())

    return "\n".join(parts)

In [30]:
# 指定章节
input_path = "output/qwen-230-instruction-data.txt"      # 你的txt格式csv文件
output_path = "output/qwen-230-instruction-data.jsonl"  # 输出jsonl文件

In [31]:
with open(input_path, "r", encoding="utf-8") as f_in, \
     open(output_path, "w", encoding="utf-8") as f_out:

    for line in f_in:
        line = line.strip()
        if not line:
            continue

        fields = line.split(";")

        if len(fields) != 9:
            print("字段数量不对，跳过：", line)
            continue

        知识点, 难度, 题型, 是否涉及运算, 认知层级, 题目, 选项, 答案, 参考知识 = fields

        # 处理选项换行
        formatted_options = format_options(选项)

        # instruction
        instruction = (
            f"根据以下条件生成试题：\n"
            f"知识点：{知识点}\n"
            f"难度：{难度}\n"
            f"运算：{是否涉及运算}\n"
            f"认知层级：{认知层级}\n"
            f"参考知识：{参考知识}"
        )

        # response
        response = (
            f"【{难度}题 | {认知层级}题 | {format_calculation_flag(是否涉及运算)}】\n"
            f"【题目】\n"
            f"{题目}\n"
            f"{formatted_options}\n"
            f"【答案】\n"
            f"{答案}"
        )

        data = {
            "instruction": instruction,
            "response": response
        }

        # 写入一行json
        f_out.write(json.dumps(data, ensure_ascii=False) + "\n")

print("转换完成。")

转换完成。


In [33]:
# 若干个jsonl文件合并-->一个jsonl文件。
input_files = [
    "output/ds-210-instruction-data.jsonl",
    "output/ds-220-instruction-data.jsonl",
    "output/ds-230-instruction-data.jsonl",
    "output/qwen-210-instruction-data.jsonl",
    "output/qwen-220-instruction-data.jsonl",
    "output/qwen-230-instruction-data.jsonl"
]

output_file = "output/ds_qwen_2_all_0_instruction_data_3000.jsonl"

with open(output_file, "w", encoding="utf-8") as fout:
    for file in input_files:
        with open(file, "r", encoding="utf-8") as fin:
            for line in fin:
                line = line.rstrip("\n")
                if line:  # 跳过空行
                    fout.write(line + "\n")

print("合并完成。")

合并完成。
